### Step 1: Install necesscary packages

In [30]:
# !pip install matplotlib
# !pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [31]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length = 64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [32]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [33]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Task 1: Generate deterministic arithmetic and algebra preference pairs

Split the dataset equally between arithmetic and algebra. Within each half,
allocate equal numbers to addition, subtraction, multiplication, and exact
division. Within each algebra operation, balance x in the left and right operand
positions (counts differ by at most one). For example, both `x/3=4,x=?` and
`12/x=4,x=?` are distinct samples.

The current 1,000-pair run produces 500 arithmetic and 500 algebra questions,
125 per operation within each half. Set `PAIR_COUNT = 100_000` for 50,000 per
half and 12,500 per operation. The count must be a multiple of eight.

Base numbers are sampled from 0-199, with nonzero constraints where needed.
Division is built from a divisor and integer quotient, so the dividend may exceed
199. Answers are integers; subtraction may produce negative results. Reject
division by zero and algebra with nonunique solutions: multiplication's known
coefficient must be nonzero, and unknown-divisor division uses a nonzero quotient.

Positives use the handout's answer-and-explanation structure. Negatives use
`Sorry, I do not know.`. A fixed seed makes generation reproducible, with no
model inference or natural-language prompt variations. Deduplicate by question
and skip strings exceeding 64 characters including four trailing newlines.
Length filtering favours shorter examples. Print category, operation, and unknown
position counts to verify coverage.

Run Steps 2-4, then these cells and Step 5 in order. Task 1 itself only needs
the tokenizer from Step 2. Rerunning replaces the JSON and timing results.

Implementation assistance: OpenAI Codex.

In [37]:
from pathlib import Path
from collections import Counter

max_length = 64
PAIR_COUNT = 300_000  # Change to 100_000 for the full dataset.

PAIR_SEED = 42
PAIR_PATH = Path("pos_neg_pairs.json")  # Run from dpo/.
TIMING_PATH = Path("task1_timing.json")
PAIR_CATEGORIES = ("arithmetic", "algebra")
PAIR_OPERATIONS = ("+", "-", "*", "/")
NEGATIVE_RESPONSE = "Sorry, I do not know."
RESPONSE_LIMIT = max_length - len("\n\n\n\n")
assert PAIR_COUNT > 0 and PAIR_COUNT % 8 == 0, "Use a positive multiple of 8."
print(f"Training sequence length: {max_length}; response budget: {RESPONSE_LIMIT}")

Training sequence length: 64; response budget: 60


In [38]:
def make_question(category, op, rng, unknown_position=None):
    a, b = rng.randint(0, 199), rng.randint(0, 199)
    if category == "algebra" and op == "*":
        # The coefficient multiplying x must not be zero.
        if unknown_position == "left":
            b = rng.randint(1, 199)
        else:
            a = rng.randint(1, 199)
    if op == "/":
        b = rng.randint(1, 199)  # Divisor.
        quotient = rng.randint(1 if unknown_position == "right" else 0, 199)
        a = b * quotient
        result = quotient
    elif op == "+":
        result = a + b
    elif op == "-":
        result = a - b
    elif op == "*":
        result = a * b
    else:
        raise ValueError(f"Unsupported operation: {op}")

    if category == "arithmetic":
        return f"{a}{op}{b}=?", result, f"{a}{op}{b}={result}"
    if category != "algebra" or unknown_position not in ("left", "right"):
        raise ValueError("Algebra requires a left or right unknown position.")
    if unknown_position == "left":
        inverse = {"+": f"{result}-{b}", "-": f"{result}+{b}",
                   "*": f"{result}/{b}", "/": f"{result}*{b}"}[op]
        return f"x{op}{b}={result},x=?", a, f"{inverse}={a}"
    inverse = {"+": f"{result}-{a}", "-": f"{a}-{result}",
               "*": f"{result}/{a}", "/": f"{a}/{result}"}[op]
    return f"{a}{op}x={result},x=?", b, f"{inverse}={b}"


def build_questions():
    rng = random.Random(PAIR_SEED)
    questions, seen = [], set()
    per_operation = PAIR_COUNT // 8
    for category in PAIR_CATEGORIES:
        for op in PAIR_OPERATIONS:
            for index in range(per_operation):
                position = ("left" if index % 2 == 0 else "right") if category == "algebra" else None
                for _ in range(10_000):
                    prompt, answer, calculation = make_question(category, op, rng, position)
                    expression, result = calculation.split("=")
                    positive = f"{prompt} The answer is {answer} because {expression} equals {result}."
                    negative = f"{prompt} {NEGATIVE_RESPONSE}"
                    if prompt in seen or max(len(positive), len(negative)) > RESPONSE_LIMIT:
                        continue
                    assert set(positive + negative) <= set(stoi), "Unsupported tokenizer character."
                    questions.append(dict(category=category, operation=op,
                                          unknown_position=position, prompt=prompt,
                                          answer=answer, calculation=calculation,
                                          positive=positive, negative=negative))
                    seen.add(prompt)
                    break
                else:
                    raise RuntimeError(
                        f"Cannot fill {category}/{op}/{position}; review ranges and length limit."
                    )
    rng.shuffle(questions)
    return questions


def save_pairs(pairs):
    temporary = PAIR_PATH.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(pairs, indent=2), encoding="utf-8")
    temporary.replace(PAIR_PATH)

In [39]:
started = time.perf_counter()
questions = build_questions()
pairs = [{"negative": q["negative"], "positive": q["positive"]} for q in questions]
assert len(pairs) == PAIR_COUNT
assert len({q["prompt"] for q in questions}) == PAIR_COUNT
assert all(len(text) + 4 <= max_length for pair in pairs for text in pair.values())
assert all(set(text) <= set(stoi) for pair in pairs for text in pair.values())
coverage = dict(Counter(q["category"] for q in questions))
assert all(count == PAIR_COUNT // 2 for count in coverage.values())
operation_coverage = {
    category: dict(Counter(q["operation"] for q in questions if q["category"] == category))
    for category in PAIR_CATEGORIES
}
position_coverage = {
    op: dict(Counter(q["unknown_position"] for q in questions
                     if q["category"] == "algebra" and q["operation"] == op))
    for op in PAIR_OPERATIONS
}
assert all(count == PAIR_COUNT // 8 for counts in operation_coverage.values() for count in counts.values())
assert all(abs(counts["left"] - counts.get("right", 0)) <= 1 for counts in position_coverage.values())
save_pairs(pairs)
elapsed_seconds = time.perf_counter() - started
timing = {
    "method": "deterministic arithmetic with fixed refusal negatives",
    "pairs": len(pairs), "seed": PAIR_SEED, "category_counts": coverage,
    "operation_counts": operation_coverage, "unknown_position_counts": position_coverage,
    "max_length": max_length, "model_inference_calls": 0,
    "elapsed_seconds": elapsed_seconds,
}
TIMING_PATH.write_text(json.dumps(timing, indent=2), encoding="utf-8")
print("Question coverage:", coverage)
print("Operation coverage:", operation_coverage)
print("Algebra unknown positions:", position_coverage)
print(f"Saved {len(pairs):,} pairs to {PAIR_PATH.resolve()}")
print("Negative response:", NEGATIVE_RESPONSE)
print(f"Generation and JSON saving: {elapsed_seconds:.3f} seconds; no model inference.")
print(json.dumps(pairs[:5], indent=2))

Question coverage: {'arithmetic': 150000, 'algebra': 150000}
Operation coverage: {'arithmetic': {'-': 37500, '/': 37500, '+': 37500, '*': 37500}, 'algebra': {'-': 37500, '*': 37500, '/': 37500, '+': 37500}}
Algebra unknown positions: {'+': {'right': 18750, 'left': 18750}, '-': {'right': 18750, 'left': 18750}, '*': {'left': 18750, 'right': 18750}, '/': {'right': 18750, 'left': 18750}}
Saved 300,000 pairs to C:\Users\yuhen\Documents\GithubRepos\NanoGPT-Math\dpo\pos_neg_pairs.json
Negative response: Sorry, I do not know.
Generation and JSON saving: 6.037 seconds; no model inference.
[
  {
    "negative": "99-94=? Sorry, I do not know.",
    "positive": "99-94=? The answer is 5 because 99-94 equals 5."
  },
  {
    "negative": "178-x=157,x=? Sorry, I do not know.",
    "positive": "178-x=157,x=? The answer is 21 because 178-157 equals 21."
  },
  {
    "negative": "x*18=1026,x=? Sorry, I do not know.",
    "positive": "x*18=1026,x=? The answer is 57 because 1026/18 equals 57."
  },
  {
   

### Step 5: Load Data (**students are required to complete this part!**)

In [46]:
# Load Task 1 pairs (working directory: dpo/).
with open("pos_neg_pairs.json", "r", encoding="utf-8") as f:
    all_lines = json.load(f)

# Shuffle reproducibly, keeping each positive-negative pair together.
random.Random(42).shuffle(all_lines)
val_lines = all_lines[:7500]
test_lines = all_lines[7500:15000]
train_lines = all_lines[15000:]
lines = train_lines  # Steps 6-7 use only training pairs.

print(f"Training: {len(train_lines):,}; validation: {len(val_lines):,}; test: {len(test_lines):,}")


Training: 285,000; validation: 7,500; test: 7,500


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
# get_batches drops the final incomplete batch.
steps_per_epoch = len(lines) // batch_size
total_steps = epochs * steps_per_epoch
if total_steps <= 0:
    raise ValueError("Training requires at least one full batch and a positive epoch count.")

min_lr = 1e-5
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)
# In Step 7, call scheduler.step() after each optimizer.step().
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=min_lr
)


### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples: 
        # ...
        # neg_logprob
        # pos_logprob 
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        # ...
        ###########################################################
        gpt.train()
        optimizer.zero_grad(set_to_none=True)
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")


### Step 8: Begin testing (**students are required to complete this part!**)

In [ ]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################